# Week 5 — Track B: the same corpus, through managed RAG

**SWE 485 · Advanced Software Engineering with AI · King Saud University**

Fifteen minutes. You will write **no retrieval code at all**. Gemini's File
Search tool does the chunking, the embedding, the vector store and the k-NN
search for you, and hands the model the results.

This track carries no marks of its own. It exists so that you can answer one
question at the end, which the examiner may well ask you:

> *You ran this through File Search too. Could you have diagnosed that failure
> here? Why not?*

---


## Before you run the next cell — read this

Uploading to a File Search store **sends this corpus to Google's servers**.

For `minibank/` that is fine. It is a teaching fixture with a fake signing key
in it. It would **not** be fine for your employer's codebase, and that is
exactly the point section 7 makes about where a code assistant's index actually
lives — except that here *you* are the one pressing the button.

Two further facts worth holding on to:

- Your API key grants access to everything you have uploaded to your stores.
  That is a stronger claim than it first sounds.
- Nothing deletes the store for you. **The last cell of this notebook deletes
  it. Run that cell.** Notice that you had to remember to.


In [ ]:
%pip install -U -q "google-genai"

In [ ]:
import pathlib, time, fnmatch

from google import genai
from google.genai import types

import config

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    import os
    GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

client = genai.Client(api_key=GEMINI_API_KEY)

TEAM = "team00"          # <-- change this, so stores do not collide
print("chat model:", config.CHAT_MODEL)

## C1 — Create a store and upload the corpus

We upload **the same files Track A indexed** — same `.ragignore`, same
exclusions — so that the comparison is like for like. Anything else and you are
comparing two corpora rather than two retrievers.


In [ ]:
store = client.file_search_stores.create(
    config=types.CreateFileSearchStoreConfig(
        display_name=f"swe485-w05-{TEAM}"
    )
)
print("created:", store.name)

In [ ]:
root = pathlib.Path(config.CORPUS_DIR)

patterns = []
for line in (root / ".ragignore").read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#"):
        patterns.append(line)


def ignored(rel):
    return any(
        fnmatch.fnmatch(rel, p) or fnmatch.fnmatch(rel, p.rstrip("/") + "/*")
        for p in patterns
    )


uploaded = []
for p in sorted(root.rglob("*")):
    if not p.is_file() or p.suffix not in config.INDEXABLE_SUFFIXES:
        continue
    rel = str(p.relative_to(root))
    if ignored(rel) or p.name == "__init__.py":
        continue

    op = client.file_search_stores.upload_to_file_search_store(
        file_search_store_name=store.name,
        file=str(p),
        config=types.UploadToFileSearchStoreConfig(display_name=rel),
    )
    while not (op := client.operations.get(op)).done:
        time.sleep(1)
    uploaded.append(rel)
    print("  uploaded", rel)

print(f"\n{len(uploaded)} files in the store")

## C2 — Ask the same questions

No embedding call. No cosine. No sort. You pass the store as a tool and the
answer comes back grounded.


In [ ]:
def managed_answer(query, top_k=None):
    top_k = config.K if top_k is None else top_k
    response = client.models.generate_content(
        model=config.CHAT_MODEL,
        contents=query,
        config=types.GenerateContentConfig(
            tools=[types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=[store.name],
                    top_k=top_k,
                )
            )]
        ),
    )

    print(f"QUERY: {query}\n")
    print(response.text)

    grounding = response.candidates[0].grounding_metadata
    print("\n" + "-" * 72)
    if grounding and grounding.grounding_chunks:
        print(f"{len(grounding.grounding_chunks)} chunks retrieved:\n")
        for i, ch in enumerate(grounding.grounding_chunks, 1):
            ctx = ch.retrieved_context
            print(f"  {i}. {ctx.title}")
            print(f"     {ctx.text[:120].strip()}...")
    else:
        print("no grounding metadata returned")
    return response


r0 = managed_answer("How does login work?")

In [ ]:
r4 = managed_answer("What happens when a charge fails?")

### Observation 1 — does it make the same mistake?

Q4 is the "wrong retrieval" trap: `refund.py` and `charges.py` share a great deal
of vocabulary, and in Track A one of them outranked the other.

Compare. Either answer is interesting — and if the managed service got it
*right*, that is worth a sentence about why, not a shrug.


### Observation 2 — find the score

`grounding_chunks` gives you the chunk text, the source title, and a page
number. Go and look for the similarity score.

Take a minute. Print the whole object if you want:


In [ ]:
g = r4.candidates[0].grounding_metadata
print(g.grounding_chunks[0])

You will not find one.

That is the observation, and it is the reason Track A is the graded track. In
Track A you could read the margin between rank 1 and rank 2 straight off the
score table and say "this failed by 0.02, a re-ranker fixes it." Here you can
see *that* the wrong chunk came back, and you can see nothing at all about *why*
or *by how much*.

Now answer this in one sentence, and be ready to say it out loud:

> **With Track B alone, how would you have diagnosed Q4?**

The same argument applies to chunking. File Search chunked these files for you,
and you cannot see where it cut. So failure mode 1 — the one you found in Track A
before running a single query, just by reading the chunk table — is not
diagnosable by inspection here at all.


### Observation 3 — `top_k` is your `k`

`top_k` is the same *k* as the `sorted(...)[:k]` you wrote in Track A. One is a
parameter you tune blind; the other is a line you wrote and can reason about.


In [ ]:
print("#" * 30, "top_k = 1", "#" * 30)
_ = managed_answer("What happens when a charge fails?", top_k=1)

print("\n\n" + "#" * 30, "top_k = 5", "#" * 30)
_ = managed_answer("What happens when a charge fails?", top_k=5)

---

## C3 — Clean up

Delete the store. Nothing else will.


In [ ]:
client.file_search_stores.delete(name=store.name, config={"force": True})
print("deleted:", store.name)

print("\nremaining stores on this key:")
for s in client.file_search_stores.list():
    print("  ", s.name)

---

## What to take from this

Fill this in before the assessment. It is four sentences, not an essay.

| | Track A — by hand | Track B — managed |
|---|---|---|
| Lines of retrieval code you wrote | | |
| Time to first grounded answer | | |
| Can you see the similarity scores? | | |
| Can you see where the chunks were cut? | | |
| Where does the index live? | | |
| Which failure could you diagnose, and which could you only observe? | | |

Managed RAG is faster to stand up and harder to debug. Both halves of that
sentence are true. Engineering judgment is knowing which half matters for the
project in front of you — and that judgment is what Week 8 and Week 9 will keep
asking you for.
